In [5]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from qutip import Qobj, basis, entropy_vn, qeye, sesolve, sigmax, sigmay, sigmaz, tensor
from scipy.linalg import eigh, svd, svdvals
from scipy.optimize import minimize_scalar


# -----------------------------------------------------------------------------
# Model and numerical settings
# -----------------------------------------------------------------------------


@dataclass(frozen=True)
class Parameters:
    """All energies are quoted in units of Omega_e."""

    J_bonds: tuple[float, ...] = (1.0, 0.03, -0.02, 0.025)
    c: tuple[float, ...] = (1.0, -1.0, 0.0, 0.0, 0.0)
    d: tuple[float, ...] = (0.0, 0.0, 1.0, 0.0, 0.0)
    Omega_e: float = 1.0
    Omega_a: float = 0.98916618
    lambda_g: float = 4.20099971
    lambda_e: float = 4.85091927


P = Parameters()
N_SOURCE_SPINS = 5
SOURCE_DIM = 2**N_SOURCE_SPINS
HARVESTER_DIM = 5

# The first, nearly perfect maximum is near Omega_e T = 0.5288.
T_SCAN_MAX = 1.20
N_TIME_SCAN = 1201
N_TRAJECTORY = 501

EULER_ANGLES = [
    (0.0, 0.0, 0.0),
    (0.2, 0.7, 1.1),
    (1.0, 0.4, 2.2),
    (2.0, 1.3, 0.5),
]


# -----------------------------------------------------------------------------
# Operator construction
# -----------------------------------------------------------------------------


def qsum(operators: list[Qobj]) -> Qobj:
    """Sum a nonempty list of Qobj instances without an integer initializer."""

    out = operators[0]
    for operator in operators[1:]:
        out = out + operator
    return out


def source_site_operator(single_spin_operator: Qobj, site: int) -> Qobj:
    factors = [qeye(2) for _ in range(N_SOURCE_SPINS)]
    factors[site] = single_spin_operator
    return tensor(factors)


def levi_civita(i: int, j: int, k: int) -> int:
    if len({i, j, k}) < 3:
        return 0
    return 1 if (i, j, k) in {(0, 1, 2), (1, 2, 0), (2, 0, 1)} else -1


def build_model(parameters: Parameters = P) -> dict[str, object]:
    # Source operators S_i^alpha = sigma_i^alpha / 2.
    single_spin = [sigmax() / 2.0, sigmay() / 2.0, sigmaz() / 2.0]
    S = [
        [source_site_operator(single_spin[alpha], site) for site in range(N_SOURCE_SPINS)]
        for alpha in range(3)
    ]
    J_s = [qsum(S[alpha]) for alpha in range(3)]
    J_s_squared = qsum([generator * generator for generator in J_s])

    H_s_terms: list[Qobj] = []
    for site, coupling in enumerate(parameters.J_bonds):
        bond = qsum([S[alpha][site] * S[alpha][site + 1] for alpha in range(3)])
        H_s_terms.append(coupling * bond)
    H_s = qsum(H_s_terms)

    T_s = [
        qsum([parameters.c[site] * S[alpha][site] for site in range(N_SOURCE_SPINS)])
        for alpha in range(3)
    ]
    W_s = [
        qsum([parameters.d[site] * S[alpha][site] for site in range(N_SOURCE_SPINS)])
        for alpha in range(3)
    ]

    # Harvester basis: |g>, |a,x>, |a,y>, |a,z>, |e>.
    h_kets = [basis(HARVESTER_DIM, index) for index in range(HARVESTER_DIM)]
    ket_g = h_kets[0]
    ket_a = h_kets[1:4]
    ket_e = h_kets[4]

    P_g = ket_g * ket_g.dag()
    P_a = qsum([ket * ket.dag() for ket in ket_a])
    P_e = ket_e * ket_e.dag()

    D_g = [ket_a[alpha] * ket_g.dag() + ket_g * ket_a[alpha].dag() for alpha in range(3)]
    D_e = [ket_a[alpha] * ket_e.dag() + ket_e * ket_a[alpha].dag() for alpha in range(3)]

    # (J_h^gamma)_{alpha,beta} = -i epsilon_{gamma,alpha,beta}
    # on the Cartesian triplet and zero on |g>, |e>.
    J_h: list[Qobj] = []
    for gamma in range(3):
        matrix = np.zeros((HARVESTER_DIM, HARVESTER_DIM), dtype=complex)
        for alpha in range(3):
            for beta in range(3):
                matrix[1 + alpha, 1 + beta] = -1j * levi_civita(gamma, alpha, beta)
        J_h.append(Qobj(matrix, dims=[[HARVESTER_DIM], [HARVESTER_DIM]]))

    I_h = qeye(HARVESTER_DIM)
    I_s = tensor([qeye(2) for _ in range(N_SOURCE_SPINS)])
    H_h = parameters.Omega_a * P_a + parameters.Omega_e * P_e

    H_int = qsum(
        [
            parameters.lambda_g * tensor(D_g[alpha], T_s[alpha])
            + parameters.lambda_e * tensor(D_e[alpha], W_s[alpha])
            for alpha in range(3)
        ]
    )
    H = tensor(H_h, I_s) + tensor(I_h, H_s) + H_int

    J_total = [tensor(J_h[alpha], I_s) + tensor(I_h, J_s[alpha]) for alpha in range(3)]

    return {
        "H": H,
        "H_s": H_s,
        "H_h": H_h,
        "J_s": J_s,
        "J_s_squared": J_s_squared,
        "J_h": J_h,
        "J_total": J_total,
        "ket_g": ket_g,
        "ket_e": ket_e,
        "P_g_full": tensor(P_g, I_s),
        "P_a_full": tensor(P_a, I_s),
        "P_e_full": tensor(P_e, I_s),
    }


# -----------------------------------------------------------------------------
# Transfer operator and seed search
# -----------------------------------------------------------------------------


def diagonalize_for_transfer(H: Qobj):
    """Diagonalize H once and return an efficient A_eg(t) constructor."""

    eigenvalues, eigenvectors = eigh(H.full(), check_finite=False)
    g_slice = slice(0, SOURCE_DIM)
    e_slice = slice(4 * SOURCE_DIM, 5 * SOURCE_DIM)

    V_row = eigenvectors[e_slice, :]
    V_col = eigenvectors.conj().T[:, g_slice]

    def A_eg(time: float) -> np.ndarray:
        phases = np.exp(-1j * eigenvalues * time)
        return (V_row * phases) @ V_col

    return eigenvalues, eigenvectors, A_eg


def find_time_and_seed(A_eg, J_s_z: np.ndarray):
    """Maximize s_max[A_eg(t)] and select a reproducible seed.

    SU(2) symmetry makes the leading singular value degenerate in magnetic
    quantum number.  Within that dominant subspace, we form an equal coherent
    superposition of the extremal J_s^z eigenvectors.  This remains a dominant
    right singular vector but makes the symmetry breaking explicit.
    """

    def largest_singular_value(time: float) -> float:
        return float(svdvals(A_eg(time), check_finite=False)[0])

    scan_times = np.linspace(0.0, T_SCAN_MAX, N_TIME_SCAN)
    scan_values = np.array([largest_singular_value(time) for time in scan_times])

    # Refine all sampled local maxima, then retain the best one.  This is more
    # robust than refining only the largest grid point.
    local_maxima = np.flatnonzero(
        (scan_values[1:-1] >= scan_values[:-2])
        & (scan_values[1:-1] >= scan_values[2:])
    ) + 1
    if len(local_maxima) == 0:
        local_maxima = np.array([int(np.argmax(scan_values))])

    candidates: list[tuple[float, float]] = []
    for index in local_maxima:
        left = scan_times[max(0, index - 1)]
        right = scan_times[min(len(scan_times) - 1, index + 1)]
        result = minimize_scalar(
            lambda time: -largest_singular_value(time),
            bounds=(left, right),
            method="bounded",
            options={"xatol": 1.0e-14},
        )
        candidates.append((-float(result.fun), float(result.x)))

    s_max, T_star = max(candidates, key=lambda pair: pair[0])
    A_star = A_eg(T_star)
    _, singular_values, Vh = svd(A_star, full_matrices=False, check_finite=False)

    # Isolate the SU(2)-degenerate leading right-singular subspace.
    # The exact magnetic degeneracy is resolved only at roundoff level.  A
    # tighter tolerance avoids merging a distinct, accidentally near-degenerate
    # singular multiplet with it.
    tolerance = 1.0e-12
    multiplicity = int(np.count_nonzero(np.abs(singular_values - singular_values[0]) < tolerance))
    V_top = Vh.conj().T[:, :multiplicity]

    projected_Jz = V_top.conj().T @ J_s_z @ V_top
    _, magnetic_vectors = eigh(projected_Jz, check_finite=False)
    psi_low = V_top @ magnetic_vectors[:, 0]
    psi_high = V_top @ magnetic_vectors[:, -1]
    psi_seed = (psi_low + psi_high) / np.sqrt(2.0)
    psi_seed /= np.linalg.norm(psi_seed)

    # Fix an irrelevant global phase to make printed amplitudes reproducible.
    pivot = int(np.argmax(np.abs(psi_seed)))
    psi_seed *= np.exp(-1j * np.angle(psi_seed[pivot]))

    fidelity = float(np.linalg.norm(A_star @ psi_seed) ** 2)
    return {
        "T_star": T_star,
        "s_max": s_max,
        "fidelity": fidelity,
        "A_star": A_star,
        "singular_values": singular_values,
        "leading_multiplicity": multiplicity,
        "psi_seed": psi_seed,
        "scan_times": scan_times,
        "scan_fidelity": scan_values**2,
    }


# -----------------------------------------------------------------------------
# SU(2) orbit and dynamical checks
# -----------------------------------------------------------------------------


def source_rotation(J_s: list[Qobj], angles: tuple[float, float, float]) -> Qobj:
    alpha, beta, gamma = angles
    return (
        (-1j * alpha * J_s[2]).expm()
        * (-1j * beta * J_s[1]).expm()
        * (-1j * gamma * J_s[2]).expm()
    )


def source_ket(vector: np.ndarray) -> Qobj:
    return Qobj(
        np.asarray(vector, dtype=complex).reshape((SOURCE_DIM, 1)),
        dims=[[2] * N_SOURCE_SPINS, [1] * N_SOURCE_SPINS],
    )


def relative_frobenius_norm(operator: Qobj, reference: Qobj) -> float:
    return float(np.linalg.norm(operator.full()) / np.linalg.norm(reference.full()))


def run_simulation(output_figure: Path) -> None:
    model = build_model(P)
    H: Qobj = model["H"]
    H_s: Qobj = model["H_s"]
    J_s: list[Qobj] = model["J_s"]
    J_total: list[Qobj] = model["J_total"]

    if not H.isherm:
        raise RuntimeError("The constructed Hamiltonian is not Hermitian.")

    symmetry_errors = [
        relative_frobenius_norm(H * generator - generator * H, H) for generator in J_total
    ]

    eigenvalues, eigenvectors, A_eg = diagonalize_for_transfer(H)
    seed_data = find_time_and_seed(A_eg, J_s[2].full())
    T_star = seed_data["T_star"]
    F_seed = seed_data["fidelity"]
    A_star = seed_data["A_star"]
    psi_seed_vector = seed_data["psi_seed"]
    psi_seed = source_ket(psi_seed_vector)

    # Source diagnostics.
    J2_expectation = float(np.real((psi_seed.dag() * model["J_s_squared"] * psi_seed)))
    Jz_expectation = float(np.real((psi_seed.dag() * J_s[2] * psi_seed)))
    Jz2_expectation = float(np.real((psi_seed.dag() * J_s[2] ** 2 * psi_seed)))
    Jz_variance = Jz2_expectation - Jz_expectation**2

    # Verify A_eg R_s = R_s A_eg and orbit fidelity invariance.
    M_star = A_star.conj().T @ A_star
    orbit_rows: list[tuple[tuple[float, float, float], float, float, float]] = []
    rotated_source_kets: list[Qobj] = []
    for angles in EULER_ANGLES:
        rotation = source_rotation(J_s, angles)
        psi_rotated = rotation * psi_seed
        vector = psi_rotated.full().ravel()
        fidelity = float(np.real(np.vdot(vector, M_star @ vector)))
        overlap = float(abs(psi_seed.overlap(psi_rotated)) ** 2)
        covariance_error = float(
            np.linalg.norm(A_star @ rotation.full() - rotation.full() @ A_star)
            / np.linalg.norm(A_star)
        )
        orbit_rows.append((angles, fidelity, overlap, covariance_error))
        rotated_source_kets.append(psi_rotated)

    # Exact unitary propagation at T_star for leakage and entropy diagnostics.
    initial_vector = np.zeros(HARVESTER_DIM * SOURCE_DIM, dtype=complex)
    initial_vector[:SOURCE_DIM] = psi_seed_vector
    final_vector = eigenvectors @ (
        np.exp(-1j * eigenvalues * T_star) * (eigenvectors.conj().T @ initial_vector)
    )
    final_ket = Qobj(
        final_vector.reshape((-1, 1)),
        dims=[[HARVESTER_DIM] + [2] * N_SOURCE_SPINS, [1] * (N_SOURCE_SPINS + 1)],
    )
    rho_h_final = final_ket.ptrace(0)
    final_entropy = float(entropy_vn(rho_h_final))
    P_g_final = float(np.real((final_ket.dag() * model["P_g_full"] * final_ket)))
    P_a_final = float(np.real((final_ket.dag() * model["P_a_full"] * final_ket)))
    P_e_final = float(np.real((final_ket.dag() * model["P_e_full"] * final_ket)))

    # Conditional final source state and the source-energy decrease.
    eta_vector = A_star @ psi_seed_vector / np.sqrt(F_seed)
    E_source_initial = float(np.real(np.vdot(psi_seed_vector, H_s.full() @ psi_seed_vector)))
    E_source_conditional = float(np.real(np.vdot(eta_vector, H_s.full() @ eta_vector)))

    print("\nSU(2) Noether-DEH five-spin simulation")
    print("=" * 48)
    print(f"Hermitian H:                         {H.isherm}")
    print("Relative commutator norms [x,y,z]:  " + ", ".join(f"{x:.3e}" for x in symmetry_errors))
    print(f"Omega_e T_star:                     {P.Omega_e * T_star:.12f}")
    print(f"s_max[A_eg(T_star)]:                {seed_data['s_max']:.12f}")
    print(f"F_e(T_star, psi_star):              {F_seed:.12f}")
    print(f"epsilon = 1-F_e:                    {1.0 - F_seed:.3e}")
    print(f"Leading singular multiplicity:      {seed_data['leading_multiplicity']}")
    print(f"<J_s^2>:                             {J2_expectation:.12f}")
    print(f"Var(J_s^z):                          {Jz_variance:.12f}")
    print(f"Final populations (g, aux, e):      ({P_g_final:.3e}, {P_a_final:.3e}, {P_e_final:.12f})")
    print(f"Final harvester entropy (nats):      {final_entropy:.3e}")
    print(f"Conditional source energy decrease: {E_source_initial - E_source_conditional:.12f}")

    print("\nEuler angles (alpha,beta,gamma) | F_e | seed overlap^2 | covariance error")
    for angles, fidelity, overlap, covariance_error in orbit_rows:
        print(
            f"{str(angles):>25s} | {fidelity:.12f} | {overlap:.9f} | {covariance_error:.3e}"
        )

    # Time-domain evolution, analogous to the original three-spin code.
    trajectory_times = np.linspace(0.0, 1.35 * T_star, N_TRAJECTORY)
    expectation_operators = [model["P_g_full"], model["P_a_full"], model["P_e_full"]]
    trajectories: list[np.ndarray] = []
    population_reference = None
    for psi_rotated in rotated_source_kets:
        result = sesolve(
            H,
            tensor(model["ket_g"], psi_rotated),
            trajectory_times,
            e_ops=expectation_operators,
            options={"atol": 1.0e-10, "rtol": 1.0e-10, "nsteps": 10000},
        )
        populations = np.asarray(result.expect, dtype=float)
        trajectories.append(populations)
        if population_reference is None:
            population_reference = populations

    max_orbit_trajectory_error = max(
        float(np.max(np.abs(populations - population_reference))) for populations in trajectories
    )
    print(f"Maximum orbit population-curve error: {max_orbit_trajectory_error:.3e}")

    # Publication-style summary plot.
    figure, axes = plt.subplots(1, 2, figsize=(12.0, 4.5))
    line_styles = ["-", "--", "-.", ":"]
    for populations, angles, style in zip(trajectories, EULER_ANGLES, line_styles):
        label = rf"$({angles[0]:.1f},{angles[1]:.1f},{angles[2]:.1f})$"
        axes[0].plot(
            P.Omega_e * trajectory_times,
            populations[2],
            linestyle=style,
            linewidth=2.0,
            label=label,
        )
    axes[0].axvline(P.Omega_e * T_star, color="black", linewidth=1.2, linestyle=(0, (4, 3)))
    axes[0].set_xlabel(r"$\Omega_e t$")
    axes[0].set_ylabel(r"$P_e(t)$")
    axes[0].set_title("SU(2)-rotated source seeds")
    axes[0].legend(title=r"$(\alpha,\beta,\gamma)$", frameon=False, fontsize=8)

    reference = trajectories[0]
    axes[1].plot(P.Omega_e * trajectory_times, reference[0], label=r"$P_g$", linewidth=2.0)
    axes[1].plot(P.Omega_e * trajectory_times, reference[1], label=r"$P_a$", linewidth=2.0)
    axes[1].plot(P.Omega_e * trajectory_times, reference[2], label=r"$P_e$", linewidth=2.0)
    axes[1].axvline(P.Omega_e * T_star, color="black", linewidth=1.2, linestyle=(0, (4, 3)))
    axes[1].set_xlabel(r"$\Omega_e t$")
    axes[1].set_ylabel("Harvester population")
    axes[1].set_title("Boundary and auxiliary populations")
    axes[1].legend(frameon=False)

    for axis in axes:
        axis.set_xlim(P.Omega_e * trajectory_times[0], P.Omega_e * trajectory_times[-1])
        axis.set_ylim(-0.025, 1.025)
        axis.grid(alpha=0.25)

    figure.suptitle("Non-Abelian Noether-DEH: five-spin source", y=1.02)
    figure.tight_layout()
    figure.savefig(output_figure, dpi=220, bbox_inches="tight")
    print(f"Figure written to: {output_figure.resolve()}")

output = Path.cwd() / "nonabelian_su2_deh_results.png"
run_simulation(output)


SU(2) Noether-DEH five-spin simulation
Hermitian H:                         True
Relative commutator norms [x,y,z]:  0.000e+00, 0.000e+00, 0.000e+00
Omega_e T_star:                     0.528784667423
s_max[A_eg(T_star)]:                0.999992830471
F_e(T_star, psi_star):              0.999985660994
epsilon = 1-F_e:                    1.434e-05
Leading singular multiplicity:      4
<J_s^2>:                             3.749999995725
Var(J_s^z):                          2.249999997150
Final populations (g, aux, e):      (1.034e-06, 1.331e-05, 0.999985660994)
Final harvester entropy (nats):      1.797e-04
Conditional source energy decrease: 0.991652663045

Euler angles (alpha,beta,gamma) | F_e | seed overlap^2 | covariance error
          (0.0, 0.0, 0.0) | 0.999985660994 | 1.000000000 | 0.000e+00
          (0.2, 0.7, 1.1) | 0.999985660994 | 0.095555699 | 3.895e-15
          (1.0, 0.4, 2.2) | 0.999985660994 | 0.006846053 | 4.091e-15
          (2.0, 1.3, 0.5) | 0.999985660994 | 0.1931851